<a href="https://colab.research.google.com/github/annasvenbro/etudesnordiques/blob/main/merge_parquet_to_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import des paquets nécessaires

In [ ]:
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
import time

# Création des chemins

In [ ]:
DATA_DIR = Path("data/sru_batches")
OUTPUT_FILE = Path("data/sudoc_final.csv")

# Lister tous les fichiers .parquet

In [ ]:
all_files = [DATA_DIR / f for f in os.listdir(DATA_DIR) if f.endswith(".parquet")]
print(f"📂 {len(all_files)} fichiers .parquet trouvés dans {DATA_DIR}")

start_time = time.time()

# Lire et concaténer les fichiers avec barre de progression

In [ ]:
dfs = []
for f in tqdm(all_files, desc="Lecture des fichiers .parquet"):
    df = pd.read_parquet(f)
    dfs.append(df)

if dfs:
    df_full = pd.concat(dfs, ignore_index=True)
else:
    df_full = pd.DataFrame()

print(f"\nNombre total d’enregistrements : {len(df_full)}")

# Supprimer les doublons sur PPN

In [ ]:
df_full = df_full.drop_duplicates(subset="PPN")
print(f"Nombre d’enregistrements uniques : {len(df_full)}")

# Réordonner les colonnes de manière cohérente

In [ ]:
expected_columns = [
    "RCR", "PPN", "Titre", "Responsabilité principale", "Responsabilité secondaire",
    "Éditeur", "Date", "Titre original", "IdRef Auteurs", "Auteurs",
    "IdRef Traducteurs", "Traducteurs"
]

# Ajouter les colonnes manquantes si nécessaire

In [ ]:
for col in expected_columns:
    if col not in df_full.columns:
        df_full[col] = None

df_full = df_full[expected_columns]

# Sauvegarde CSV

In [ ]:
df_full.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
duration = time.time() - start_time

print(f"\n✅ CSV final créé : {OUTPUT_FILE} (temps écoulé : {duration:.2f}s)")

# Aperçu

In [ ]:
print("\nAperçu des 5 premières lignes :")
print(df_full.head())